In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

# 1. Initialize the Zero-Shot Classification Pipeline
# This will run on GPU if available, otherwise CPU
print("Loading Transformer Model for semantic validation...")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1) # change to 0 if GPU is active

df=pd.read_csv("PM_Modi_speeches.csv")

# 2. Define natural descriptions representing your 10-point framework
framework_labels = [
    "Character Integrity or Corruption accusations",
    "Personal Traits, arrogance or weakness",
    "Ridicule, Mockery or calling someone a joke",
    "Derogatory Language or insults",
    "Misconduct, Criminality or financial scams",
    "Emotional Outbursts or expressions of shame/hate",
    "Incompetence, unfitness or capability doubts",
    "Family attacks, Nepotism or dynasty politics",
    "Physical Appearance, age or health criticism",
    "Guilt by Association or corrupt alliances"
]

# 3. Apply semantic framework prediction over text samples
# Note: For large datasets, sample first or process sentence-by-sentence
sample_speeches = df['text'].fillna('').head(50).tolist() # Testing on 50 speeches

semantic_results = []
print("Evaluating text semantically against the 10-point framework...")
for text in tqdm(sample_speeches):
    if len(text.strip()) == 0:
        semantic_results.append({lbl: 0.0 for lbl in framework_labels})
        continue

    # Run the zero-shot inference
    res = classifier(text, candidate_labels=framework_labels, multi_label=True)

    # Map confidence scores back to labels
    score_dict = dict(zip(res['labels'], res['scores']))
    semantic_results.append(score_dict)

# Convert results to DataFrame and merge back with original speech tracking
df_semantic = pd.DataFrame(semantic_results)
df_upgraded = pd.concat([df.head(50).reset_index(drop=True), df_semantic], axis=1)

print("Semantic matrix extraction complete! Top scores reflect deep context matches.")
print(df_upgraded[[*framework_labels]].head())

Loading Transformer Model for semantic validation...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Evaluating text semantically against the 10-point framework...


100%|██████████| 50/50 [2:06:52<00:00, 152.24s/it]

Semantic matrix extraction complete! Top scores reflect deep context matches.
   Character Integrity or Corruption accusations  \
0                                       0.411848   
1                                       0.233086   
2                                       0.320655   
3                                       0.137426   
4                                       0.211787   

   Personal Traits, arrogance or weakness  \
0                                0.436105   
1                                0.402812   
2                                0.674965   
3                                0.317602   
4                                0.388834   

   Ridicule, Mockery or calling someone a joke  \
0                                     0.242490   
1                                     0.128677   
2                                     0.339180   
3                                     0.038875   
4                                     0.116225   

   Derogatory Language or insults  Mi

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# 1. Setup Data for Deep Learning Pipeline
# Assumes 'label' is your heuristic ground truth column from your pipeline
X = df['text'].fillna('').astype(str).tolist()
y = df['label'].tolist()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 2. Tokenization Setup
model_name = "distilbert-base-uncased" # Fast, high-performing transformer
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=256)

# Convert arrays into HuggingFace dataset format
train_ds = Dataset.from_dict({'text': X_train, 'label': y_train}).map(tokenize_function, batched=True)
val_ds = Dataset.from_dict({'text': X_val, 'label': y_val}).map(tokenize_function, batched=True)

# 3. Configure Transformer Classifier Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Compute tracking metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

# 4. Define Hyperparameters (TrainingArguments)
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

# Initialize Trainer Execution Framework
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# 5. Train Model and Run Predictions
print("Starting Transformer Fine-Tuning Process...")
trainer.train()

# Get Val predictions for comparative Analysis report
raw_preds = trainer.predict(val_ds)
y_pred = np.argmax(raw_preds.predictions, axis=-1)

print("\n=== FINE-TUNED TRANSFORMER PERFORMANCE CLASSIFICATION REPORT ===")
print(classification_report(y_val, y_pred, target_names=['Policy / Non-Attack', 'Personal Attack']))

In [ ]:
# Install BERTopic if not installed in your sandbox yet:
# !pip install bertopic

from bertopic import BERTopic
import pandas as pd

# 1. Initialize and Fit BERTopic Pipeline
print("Extracting contextual topic embeddings...")
docs = df['text'].fillna('').astype(str).tolist()
timestamps = df['year'].tolist() # Uses the years tracked in your Jargon notebook

# Create BERTopic framework instance
topic_model = BERTopic(language="english", calculate_probabilities=False, verbose=True)
topics, probs = topic_model.fit_transform(docs)

# 2. View Top Discovered Semantic Topics
print("\n--- NEW TRANSFORMER-BASED HIDDEN TOPICS ---")
print(topic_model.get_topic_info().head(10))

# 3. Compute Dynamic Topic Evolution Over Time
print("\nModeling Dynamic Evolution of Topics across election years...")
topics_over_time = topic_model.topics_over_time(docs, timestamps)

# 4. Visualize the Temporal Shifts
# Generates an interactive line trace showing topic prominence by year
fig = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=8)
fig.write_html("dynamic_topics_evolution.html")
print("Interactive Dynamic Topic Chart saved safely as 'dynamic_topics_evolution.html'.")